# 🎙️ Stream A — Acoustic Prosody Pipeline

**Lead:** Lily Wu  
**PI:** Peter Zhou  
**Dataset:** [DAIC-WOZ](https://dcapswoz.ict.usc.edu/) (~189 subjects)  

### Pipeline
```
DAIC-WOZ .wav → Whisper large-v3 → Pyannote diarization → OpenSMILE eGeMAPSv02 → VTA → stream_a_features.csv
```

### Output
One CSV file: `stream_a_features.csv` with per-subject acoustic features including the VTA (Vocal Tone of Anhedonia).

---

> ⚠️ **Runtime:** Go to `Runtime → Change runtime type → GPU (T4)` before running.

> ⚠️ **Data:** DAIC-WOZ requires a signed data use agreement. Upload audio files to your Google Drive.

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify mount
import os
assert os.path.exists('/content/drive/MyDrive'), 'Drive not mounted!'
print('✓ Google Drive mounted')

## Cell 2 — Install Dependencies

Run once per Colab session. Takes ~2-3 minutes.

In [ ]:
%%capture
# Core
!pip install openai-whisper
!pip install opensmile
!pip install soundfile librosa pandas numpy scipy tqdm

# Pyannote (speaker diarization)
!pip install pyannote.audio

# Verify
import whisper
import opensmile
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✓ All dependencies installed')
print(f'  Whisper: {whisper.__version__}')
print(f'  Device: {device}')
if device == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## Cell 3 — Configuration

**Edit the paths below** to match where you stored DAIC-WOZ audio files on your Drive.

In [ ]:
import os
import glob

# =======================================================
# CONFIGURATION — EDIT THESE
# =======================================================

# Whisper model: use 'base' for testing, 'large-v3' for real data
MODEL_SIZE = 'large-v3'

# Path to DAIC-WOZ audio files on your Google Drive
# Each subject should be a .wav file, e.g., 300_AUDIO.wav
AUDIO_DIR = '/content/drive/MyDrive/NSG_Dopaminergic_Voice/DAIC-WOZ/audio'

# Where to save results (also on Drive so they persist)
RESULTS_DIR = '/content/drive/MyDrive/NSG_Dopaminergic_Voice/Results/Stream_A'

# HuggingFace token for Pyannote (get from https://huggingface.co/settings/tokens)
# You must also accept terms at https://huggingface.co/pyannote/speaker-diarization-3.1
HF_TOKEN = ''  # Paste your token here

# Audio settings
SAMPLE_RATE = 16000
AUDIO_EXTENSIONS = ['*.wav', '*.mp3', '*.m4a']

# =======================================================
# AUTO-SETUP
# =======================================================
os.makedirs(RESULTS_DIR, exist_ok=True)

# Find all audio files
audio_files = []
for ext in AUDIO_EXTENSIONS:
    audio_files.extend(glob.glob(os.path.join(AUDIO_DIR, ext)))
audio_files = sorted(audio_files)

print(f'Model: {MODEL_SIZE}')
print(f'Audio dir: {AUDIO_DIR}')
print(f'Results dir: {RESULTS_DIR}')
print(f'Audio files found: {len(audio_files)}')

if len(audio_files) == 0:
    print('\n⚠️  No audio files found! Check your AUDIO_DIR path.')
    print(f'    Expected: {AUDIO_DIR}/<subject_id>.wav')
else:
    print(f'\nFirst 5 files:')
    for f in audio_files[:5]:
        print(f'  • {os.path.basename(f)}')

## Cell 4 — Load Models

Loads Whisper, Pyannote, and OpenSMILE into memory. This takes ~2 min on first run (downloads model weights).

In [ ]:
import whisper
import opensmile
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- Whisper ---
print(f'Loading Whisper {MODEL_SIZE}...')
whisper_model = whisper.load_model(MODEL_SIZE, device=device)
print(f'  ✓ Whisper loaded on {device}')

# --- Pyannote (Speaker Diarization) ---
diarize_pipeline = None
if HF_TOKEN:
    try:
        from pyannote.audio import Pipeline
        print('Loading Pyannote speaker diarization...')
        diarize_pipeline = Pipeline.from_pretrained(
            'pyannote/speaker-diarization-3.1',
            use_auth_token=HF_TOKEN
        )
        diarize_pipeline.to(torch.device(device))
        print('  ✓ Pyannote loaded')
    except Exception as e:
        print(f'  ⚠ Pyannote failed: {e}')
        print('  Continuing without diarization.')
else:
    print('  ℹ No HF_TOKEN — skipping Pyannote diarization.')
    print('  Pipeline will still extract features from FULL audio (no speaker separation).')

# --- OpenSMILE (eGeMAPSv02) ---
print('Loading OpenSMILE eGeMAPSv02...')
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)
print(f'  ✓ OpenSMILE loaded ({len(smile.feature_names)} features)')

print('\n🟢 All models loaded. Ready to process.')

## Cell 5 — Feature Extraction Functions

Core pipeline logic. **You should NOT need to edit this cell.**

In [ ]:
import math
import librosa
import soundfile as sf
import numpy as np
import tempfile


def extract_participant_features(audio_path, whisper_model, smile, diarize_pipeline=None):
    """
    Full pipeline for one participant:
      1. Transcribe with Whisper
      2. (Optional) Diarize with Pyannote to isolate participant speech
      3. Extract eGeMAPSv02 features with OpenSMILE
      4. Compute VTA
    
    Returns: dict of features for this participant
    """
    subject_id = os.path.splitext(os.path.basename(audio_path))[0]
    
    # ---------- Step 1: Whisper Transcription ----------
    result = whisper_model.transcribe(
        audio_path,
        language='en',
        verbose=False,
        word_timestamps=True
    )
    transcript = result.get('text', '').strip()
    segments = result.get('segments', [])
    
    # Basic transcript stats
    word_count = len(transcript.split())
    n_segments = len(segments)
    total_duration = segments[-1]['end'] if segments else 0
    
    # ---------- Step 2: Load audio for acoustic analysis ----------
    audio_data, sr = librosa.load(audio_path, sr=SAMPLE_RATE)
    
    # If diarization is available, extract only the PARTICIPANT's speech
    # (In DAIC-WOZ, the participant is typically the longer speaker)
    if diarize_pipeline is not None:
        try:
            diarization = diarize_pipeline(audio_path)
            
            # Identify speakers and their total durations
            speaker_durations = {}
            speaker_segments = {}
            for turn, _, speaker in diarization.itertracks(yield_label=True):
                dur = turn.end - turn.start
                speaker_durations[speaker] = speaker_durations.get(speaker, 0) + dur
                if speaker not in speaker_segments:
                    speaker_segments[speaker] = []
                speaker_segments[speaker].append((turn.start, turn.end))
            
            # The participant is typically the speaker who talks the most
            if speaker_durations:
                participant_speaker = max(speaker_durations, key=speaker_durations.get)
                
                # Extract only participant audio segments
                participant_audio = []
                for start, end in speaker_segments[participant_speaker]:
                    start_sample = int(start * SAMPLE_RATE)
                    end_sample = int(end * SAMPLE_RATE)
                    participant_audio.append(audio_data[start_sample:end_sample])
                
                if participant_audio:
                    audio_data = np.concatenate(participant_audio)
                    
        except Exception as e:
            print(f'    ⚠ Diarization failed for {subject_id}: {e}')
            # Continue with full audio
    
    # ---------- Step 3: OpenSMILE Feature Extraction ----------
    # Write audio to temp file (OpenSMILE requires a file path)
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
        tmp_path = tmp.name
    
    try:
        sf.write(tmp_path, audio_data, SAMPLE_RATE)
        df_features = smile.process_file(tmp_path)
        features = df_features.iloc[0].to_dict()
    except Exception as e:
        print(f'    ⚠ OpenSMILE failed for {subject_id}: {e}')
        features = {}
    finally:
        if os.path.exists(tmp_path):
            os.unlink(tmp_path)
    
    # ---------- Step 4: Extract key metrics + compute VTA ----------
    # F0 (pitch)
    f0_mean = features.get('F0semitoneFrom27.5Hz_sma3nz_amean', 0.0)
    f0_cv = features.get('F0semitoneFrom27.5Hz_sma3nz_stddevNorm', 0.0)
    f0_range = features.get('F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2', 0.0)
    
    # Loudness / Energy
    loudness_mean = features.get('loudness_sma3_amean', 0.0)
    loudness_cv = features.get('loudness_sma3_stddevNorm', 0.0)
    
    # Jitter & Shimmer (voice quality)
    jitter = features.get('jitterLocal_sma3nz_amean', 0.0)
    shimmer = features.get('shimmerLocaldB_sma3nz_amean', 0.0)
    
    # Spectral
    spectral_flux = features.get('spectralFlux_sma3_amean', 0.0)
    mfcc1 = features.get('mfcc1_sma3_amean', 0.0)
    mfcc2 = features.get('mfcc2_sma3_amean', 0.0)
    mfcc3 = features.get('mfcc3_sma3_amean', 0.0)
    mfcc4 = features.get('mfcc4_sma3_amean', 0.0)
    
    # HNR (Harmonics-to-Noise Ratio)
    hnr = features.get('HNRdBACF_sma3nz_amean', 0.0)
    
    # Speaking rate proxies
    speaking_rate = word_count / max(total_duration / 60, 0.01)  # words per minute
    
    # Pause analysis (from Whisper segments)
    pauses = []
    for i in range(1, len(segments)):
        gap = segments[i]['start'] - segments[i-1]['end']
        if gap > 0.3:  # Pauses > 300ms
            pauses.append(gap)
    
    n_pauses = len(pauses)
    mean_pause_dur = np.mean(pauses) if pauses else 0.0
    total_pause_dur = sum(pauses)
    pause_ratio = total_pause_dur / max(total_duration, 0.01)
    
    # ---------- VTA (Vocal Tone of Anhedonia) ----------
    # V_anh = -log(CV_F0 × CV_Energy)
    vta = 0.0
    if f0_cv > 0 and loudness_cv > 0:
        try:
            product = float(f0_cv) * float(loudness_cv)
            if product > 0:
                vta = -math.log(product)
        except Exception:
            pass
    
    return {
        'participant_id': subject_id,
        # Pitch
        'f0_mean': round(float(f0_mean), 4),
        'f0_cv': round(float(f0_cv), 4),
        'f0_range': round(float(f0_range), 4),
        # Loudness
        'loudness_mean': round(float(loudness_mean), 4),
        'loudness_cv': round(float(loudness_cv), 4),
        # Voice quality
        'jitter': round(float(jitter), 6),
        'shimmer': round(float(shimmer), 4),
        'hnr': round(float(hnr), 4),
        # Spectral
        'spectral_flux': round(float(spectral_flux), 4),
        'mfcc1': round(float(mfcc1), 4),
        'mfcc2': round(float(mfcc2), 4),
        'mfcc3': round(float(mfcc3), 4),
        'mfcc4': round(float(mfcc4), 4),
        # Temporal / speaking
        'speaking_rate_wpm': round(float(speaking_rate), 2),
        'n_pauses': n_pauses,
        'mean_pause_duration': round(float(mean_pause_dur), 4),
        'pause_ratio': round(float(pause_ratio), 4),
        # Composite
        'vta': round(float(vta), 4),
        # Transcript metadata
        'word_count': word_count,
        'duration_sec': round(float(total_duration), 2),
        'n_whisper_segments': n_segments,
    }


print('✓ Pipeline functions defined')

## Cell 6 — Run Pipeline on All Subjects

This is the main processing loop. Processes each audio file and saves progress incrementally.

**Estimated time:**
- 5 subjects (Hello World): ~10-15 min on T4
- 189 subjects (full run): ~6-8 hours on T4, ~3-4 hours on A100

In [ ]:
import pandas as pd
from tqdm import tqdm
import time

# =======================================================
# LIMIT — Set to a small number for testing, None for all
# =======================================================
MAX_SUBJECTS = 5  # Change to None for full run

files_to_process = audio_files[:MAX_SUBJECTS] if MAX_SUBJECTS else audio_files
print(f'Processing {len(files_to_process)} of {len(audio_files)} subjects')
print(f'Model: {MODEL_SIZE} | Device: {device}')
print('=' * 60)

all_features = []
errors = []
checkpoint_path = os.path.join(RESULTS_DIR, 'stream_a_features_checkpoint.csv')

start_time = time.time()

for i, audio_path in enumerate(tqdm(files_to_process, desc='Processing')):
    subject_id = os.path.splitext(os.path.basename(audio_path))[0]
    
    try:
        features = extract_participant_features(
            audio_path, whisper_model, smile, diarize_pipeline
        )
        all_features.append(features)
        print(f'  [{i+1}/{len(files_to_process)}] ✓ {subject_id} — VTA: {features["vta"]:.4f}')
        
    except Exception as e:
        errors.append({'subject': subject_id, 'error': str(e)})
        print(f'  [{i+1}/{len(files_to_process)}] ✗ {subject_id} — {e}')
    
    # Checkpoint every 10 subjects
    if (i + 1) % 10 == 0 and all_features:
        pd.DataFrame(all_features).to_csv(checkpoint_path, index=False)
        print(f'    💾 Checkpoint saved ({len(all_features)} subjects)')

elapsed = time.time() - start_time
print('\n' + '=' * 60)
print(f'DONE — {len(all_features)} succeeded, {len(errors)} failed')
print(f'Time: {elapsed/60:.1f} min ({elapsed/max(len(files_to_process),1):.1f}s per subject)')

if errors:
    print(f'\n⚠ Errors:')
    for e in errors:
        print(f"  • {e['subject']}: {e['error']}")

## Cell 7 — Save Results

In [ ]:
import pandas as pd

if not all_features:
    print('⚠ No features extracted! Check errors above.')
else:
    df = pd.DataFrame(all_features)
    
    # Save to Drive (persists after session ends)
    output_path = os.path.join(RESULTS_DIR, 'stream_a_features.csv')
    df.to_csv(output_path, index=False)
    print(f'✓ Saved: {output_path}')
    print(f'  Shape: {df.shape[0]} subjects × {df.shape[1]} features')
    
    # Save errors log
    if errors:
        errors_path = os.path.join(RESULTS_DIR, 'stream_a_errors.csv')
        pd.DataFrame(errors).to_csv(errors_path, index=False)
        print(f'  Errors log: {errors_path}')
    
    # Display summary
    print('\n' + '=' * 60)
    print('STREAM A — FEATURE SUMMARY')
    print('=' * 60)
    print(df.describe().round(4).to_string())
    print('\n')
    display(df.head(10))

## Cell 8 — Quick Sanity Checks

Run these to verify the output before submitting to Peter.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')

if 'df' not in dir() or df.empty:
    print('No data to plot. Run cells above first.')
else:
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle('Stream A — Acoustic Feature Distributions', fontsize=16, fontweight='bold')
    
    features_to_plot = [
        ('vta', 'VTA (Vocal Tone of Anhedonia)'),
        ('f0_mean', 'Pitch Mean (semitones)'),
        ('f0_cv', 'Pitch CV'),
        ('loudness_cv', 'Loudness CV'),
        ('speaking_rate_wpm', 'Speaking Rate (WPM)'),
        ('pause_ratio', 'Pause Ratio'),
    ]
    
    for ax, (col, title) in zip(axes.flat, features_to_plot):
        if col in df.columns:
            ax.hist(df[col], bins=20, edgecolor='white', alpha=0.8)
            ax.set_title(title, fontweight='bold')
            ax.axvline(df[col].median(), color='red', linestyle='--', label=f'Median: {df[col].median():.3f}')
            ax.legend(fontsize=9)
    
    plt.tight_layout()
    plot_path = os.path.join(RESULTS_DIR, 'stream_a_distributions.png')
    plt.savefig(plot_path, dpi=150)
    plt.show()
    print(f'✓ Plot saved: {plot_path}')
    
    # Sanity checks
    print('\n🔍 Sanity Checks:')
    print(f'  Subjects with VTA = 0: {(df["vta"] == 0).sum()} (should be rare)')
    print(f'  Subjects with 0 word count: {(df["word_count"] == 0).sum()} (should be 0)')
    print(f'  VTA range: [{df["vta"].min():.3f}, {df["vta"].max():.3f}]')
    print(f'  Pitch mean range: [{df["f0_mean"].min():.1f}, {df["f0_mean"].max():.1f}] semitones')

---

## ✅ Done!

Your output file is at:
```
Google Drive → NSG_Dopaminergic_Voice → Results → Stream_A → stream_a_features.csv
```

**Next steps:**
1. Share `stream_a_features.csv` with Peter on Slack (`#stream-a-acoustic`)
2. Peter will run QC + test correlation
3. If passed → proceed to full 189-subject extraction (change `MAX_SUBJECTS = None`)